In [ ]:
import scanpy as sc
import anndata as ad
from pathlib import Path
import sys

In [ ]:
# --- 1. Define the Root Directory ---
# This is the main directory containing all region subfolders (e.g., 'vz', 'svz')
base_dir = Path("Process_Data/bin100_3D_region")

if not base_dir.is_dir():
    print(f"Error: Root directory not found: {base_dir}")
    print("Please ensure the path is correct.")
    sys.exit(1)

# --- 2. Find all region subfolders ---
print(f"Scanning root directory: {base_dir}")
# We iterate over all items and keep only those that are directories
try:
    # p.is_dir() checks if the path is a directory
    region_folders = [p for p in base_dir.iterdir() if p.is_dir()]
except Exception as e:
    print(f"Error while scanning directory {base_dir}: {e}")
    sys.exit(1)

if not region_folders:
    print(f"No region subfolders found in {base_dir}.")
    sys.exit(0)

print(f"Found {len(region_folders)} region folders to process:")
for f in region_folders:
    print(f"  - {f.name}")
print("=" * 40)


# --- 3. Loop through each region folder ---
for target_folder in region_folders:
    # target_folder is a Path object, e.g., .../bin100_3D_region/vz
    # region_name is just the folder's name, e.g., 'vz'
    region_name = target_folder.name
    
    print(f"\n>>> Starting process for region: {region_name}")
    print(f"    Target folder: {target_folder}")

    # --- 3.1 Define the output file path ---
    output_file_name = f"combined_adata_{region_name}.h5ad"
    # Save the merged file in the parent directory (base_dir)
    output_path = base_dir / output_file_name 
    print(f"    Merged file will be saved to: {output_path}")

    # --- 3.2 Find all h5ad files to merge ---
    # glob will only find files inside the target_folder
    h5ad_files_to_merge = []
    print(f"    Scanning for .h5ad files starting with a digit...") # 提示信息
    
    for f in target_folder.glob("*.h5ad"):

        # (f.name and ... 是为了确保文件名非空)
        if f.name and f.name[0].isdigit():
            # 这个文件符合所有标准
            h5ad_files_to_merge.append(f)
        else:
            # 这个文件不是合并文件，但也不是数字开头
            print(f"     (Skipping non-numeric file: {f.name})")
            continue # 跳过，处理下一个文件

    if not h5ad_files_to_merge:
        print(f"    [Info] No .h5ad files found starting with a digit in {target_folder}. Skipping this region.")
        continue # Move to the next region folder

    print(f"    Found {len(h5ad_files_to_merge)} sample files to merge.")

    # --- 3.3 Read files and prepare for merge ---
    adata_list = []
    keys = []
    print(f"    Reading {len(h5ad_files_to_merge)} files...")
    
    all_read_success = True
    for file_path in h5ad_files_to_merge:
        try:
            # print(f"         - {file_path.name}") # (Uncomment for verbose logging)
            adata_list.append(sc.read_h5ad(file_path))
            keys.append(file_path.stem) # Use filename (without .h5ad) as the key
        except Exception as e:
            print(f"    [!! ERROR !!] Failed to read {file_path.name}: {e}. Skipping this file.")
            all_read_success = False
    
    if not adata_list:
        print(f"    [Error] No files were successfully read for region '{region_name}'. Skipping this region.")
        continue

    if not all_read_success:
        print(f"    [Warning] Some files failed to read. Merging only the successfully read files.")
        
    print(f"    File reading complete. Starting merge...")

    # --- 3.4 Concatenate ---
    # This uses the corrected code to solve the 'Observation names are not unique' warning
    try:
        combined_adata = ad.concat(
            adata_list,
            label='sample_id',     # Creates a new .obs column 'sample_id'
            keys=keys,             # Uses values from the 'keys' list (filenames) to fill 'sample_id'
            index_unique='-',      # Makes obs_names unique with the key and '-' (e.g., "9_D03...-AAACCCAAGGAG")
            join='outer'           # Keeps all genes from all files (takes the union)
        )
        
        print(f"    Region '{region_name}' merged successfully.")
        print(f"      - Total spots (n_obs): {combined_adata.n_obs}")
        print(f"      - Total genes (n_var): {combined_adata.n_vars}")
        print(f"      - Example obs_name: {combined_adata.obs_names[0]}")

    except Exception as e:
        print(f"    [!! ERROR !!] Merge failed for region '{region_name}': {e}")
        print("    Please check if files in this region are compatible (e.g., .var gene lists).")
        print("    Skipping this region.")
        continue # Move to the next region folder

    # --- 3.5 Save the combined file ---
    print(f"    Saving merged file to: {output_path}")
    try:
        # Use gzip compression to save space
        combined_adata.write_h5ad(output_path, compression='gzip')
        print(f"    Region '{region_name}' saved successfully!")
    except Exception as e:
        print(f"    [!! ERROR !!] Failed to save file: {e}")

    print(f"<<< Finished region: {region_name}")
    print("-" * 40)

print("\n===============================")
print("All region processing complete.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_region_spot_counts_vertical(data_dict, title='Total Spots per Region'):
    """
    Plots a vertical bar chart from a dictionary of data,
    with bold labels and thicker borders.
    """
    
    # --- 1. Data Preparation ---
    # Convert dictionary to Pandas Series for easy sorting
    data_series = pd.Series(data_dict)
    
    # Sort by value (spot count) in descending order for vertical bars,
    # so the tallest bars appear first (left).
    data_series = data_series.sort_values(ascending=False)
    
    # --- 2. Font and Style Settings ---
    # Set global font size and weight
    plt.rcParams['font.size'] = 14 # Base font size
    plt.rcParams['font.weight'] = 'bold' # Global bold font
    plt.rcParams['axes.titlesize'] = 18 # Title font size
    plt.rcParams['axes.labelsize'] = 14 # Axis label font size
    plt.rcParams['xtick.labelsize'] = 14 # X-tick label font size
    plt.rcParams['ytick.labelsize'] = 14 # Y-tick label font size
    plt.rcParams['figure.titleweight'] = 'bold' # Figure title bold
    plt.rcParams['axes.labelweight'] = 'bold' # Axis labels bold
    
    # --- 3. Start Plotting ---
    # Create a figure, set size (15in wide, 8in high to accommodate 18 regions horizontally)
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Draw the vertical bar chart
    # Note: x is regions, height is spot counts for vertical bars
    bars = ax.bar(data_series.index, data_series.values, color='steelblue', 
                  edgecolor='black', linewidth=1.5) # Thicker black edge

    # --- 4. Add Value Labels ---
    # Display the specific spot count at the top of each bar
    for bar in bars:
        height = bar.get_height()
        # Add text label above the bar
        ax.text(
            bar.get_x() + bar.get_width() / 2, # X-axis centered on the bar
            height + (height * 0.01),          # Position 1% above the bar
            f'{height:,}',                     # Format number, e.g., "792,527"
            ha='center',                       # Horizontal alignment
            va='bottom',                       # Vertical alignment
            fontsize=14,
            weight='bold',                      # Make value labels bold
            rotation=45,
        )

    # --- 5. Style the Plot ---
    ax.set_title(title, pad=20) # Title will use rcParams for size/weight
    ax.set_xlabel('Brain Region') # Label will use rcParams for size/weight
    ax.set_ylabel('Total Spots (n_obs)') # Label will use rcParams for size/weight
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right', weight='bold') # Rotate and make bold
    plt.yticks(weight='bold') # Make y-tick labels bold
    
    # Add grid for better readability (optional)
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    
    # Remove top and right spines (if desired, though you asked for "加粗边框" which implies keeping them)
    # If you want to keep all borders and just make them thicker, comment these out.
    # If "加粗边框" specifically meant the bars' edges, then these lines are fine.
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5) # Thicken left spine
    ax.spines['bottom'].set_linewidth(1.5) # Thicken bottom spine
    
    # Adjust Y-axis limit to make space for labels above the bars
    max_value = data_series.max()
    ax.set_ylim(0, max_value * 1.15) 
    
    # Adjust layout automatically to prevent labels from being cut off
    plt.tight_layout()
    
    # --- 6. Save and Show ---
    save_path = "region_spot_counts_vertical_chart.png"
    plt.savefig(save_path, dpi=150)
    print(f"Chart saved to: {save_path}")
    
    # Show the plot
    plt.show()

# --- Main execution block ---
if __name__ == "__main__":
    
    # 1. Data extracted from your log
    spot_data = {
        "subthalamic nucleus": 2913,
        "CP_SP": 792527,
        "NAc": 34967,
        "LGE": 161427,
        "IZ": 576734,
        "ependymal region": 29471,
        "STR": 133633,
        "midbrain": 280789,
        "thalamus": 192083,
        "cerebellum": 52414,
        "amygdala": 28180,
        "Hindbrain": 181617,
        "VZ_GE": 22017,
        "MGE": 227992,
        "PONS": 23601,
        "HIP": 53097,
        "SVZ": 416253,
        "VZ": 236845
    }
    
    # 2. Call the function to plot
    plot_region_spot_counts_vertical(spot_data, title="Total Spot Counts for 18 Brain Regions")



In [ ]:
# --- 1. Define the Root Directory ---
# This is the main directory containing all region subfolders (e.g., 'vz', 'svz')
base_dir = Path("Process_Data/bin100_3D_region")

if not base_dir.is_dir():
    print(f"Error: Root directory not found: {base_dir}")
    print("Please ensure the path is correct.")
    sys.exit(1)

# --- 2. Find all region subfolders ---
print(f"Scanning root directory: {base_dir}")
# We iterate over all items and keep only those that are directories
try:
    # p.is_dir() checks if the path is a directory
    region_folders = [p for p in base_dir.iterdir() if p.is_dir()]
except Exception as e:
    print(f"Error while scanning directory {base_dir}: {e}")
    sys.exit(1)

if not region_folders:
    print(f"No region subfolders found in {base_dir}.")
    sys.exit(0)

print(f"Found {len(region_folders)} region folders to process:")
for f in region_folders:
    print(f"  - {f.name}")
print("=" * 40)


# --- 3. Loop through each region folder ---
for target_folder in region_folders:
    # target_folder is a Path object, e.g., .../bin100_3D_region/vz
    # region_name is just the folder's name, e.g., 'vz'
    region_name = target_folder.name
    if region_name != 'cb_chp':
        continue
    print(f"\n>>> Starting process for region: {region_name}")
    print(f"    Target folder: {target_folder}")

    # --- 3.1 Define the output file path ---
    output_file_name = f"combined_adata_{region_name}.h5ad"
    # Save the merged file in the parent directory (base_dir)
    output_path = base_dir / output_file_name 
    print(f"    Merged file will be saved to: {output_path}")

    # --- 3.2 Find all h5ad files to merge ---
    # glob will only find files inside the target_folder
    h5ad_files_to_merge = []
    print(f"    Scanning for .h5ad files starting with a digit...") # 提示信息
    
    for f in target_folder.glob("*.h5ad"):

        # (f.name and ... 是为了确保文件名非空)
        if f.name and f.name[0].isdigit():
            # 这个文件符合所有标准
            h5ad_files_to_merge.append(f)
        else:
            # 这个文件不是合并文件，但也不是数字开头
            print(f"     (Skipping non-numeric file: {f.name})")
            continue # 跳过，处理下一个文件

    if not h5ad_files_to_merge:
        print(f"    [Info] No .h5ad files found starting with a digit in {target_folder}. Skipping this region.")
        continue # Move to the next region folder

    print(f"    Found {len(h5ad_files_to_merge)} sample files to merge.")

    # --- 3.3 Read files and prepare for merge ---
    adata_list = []
    keys = []
    print(f"    Reading {len(h5ad_files_to_merge)} files...")
    
    all_read_success = True
    for file_path in h5ad_files_to_merge:
        try:
            adata = sc.read_h5ad(file_path)
            adata.var_names_make_unique()
            # print(f"         - {file_path.name}") # (Uncomment for verbose logging)
            adata_list.append(adata)
            keys.append(file_path.stem) # Use filename (without .h5ad) as the key
        except Exception as e:
            print(f"    [!! ERROR !!] Failed to read {file_path.name}: {e}. Skipping this file.")
            all_read_success = False
    
    if not adata_list:
        print(f"    [Error] No files were successfully read for region '{region_name}'. Skipping this region.")
        continue

    if not all_read_success:
        print(f"    [Warning] Some files failed to read. Merging only the successfully read files.")
        
    print(f"    File reading complete. Starting merge...")

    # --- 3.4 Concatenate ---
    # This uses the corrected code to solve the 'Observation names are not unique' warning
    try:
        combined_adata = ad.concat(
            adata_list,
            label='sample_id',     # Creates a new .obs column 'sample_id'
            keys=keys,             # Uses values from the 'keys' list (filenames) to fill 'sample_id'
            index_unique='-',      # Makes obs_names unique with the key and '-' (e.g., "9_D03...-AAACCCAAGGAG")
            join='outer'           # Keeps all genes from all files (takes the union)
        )
        
        print(f"    Region '{region_name}' merged successfully.")
        print(f"      - Total spots (n_obs): {combined_adata.n_obs}")
        print(f"      - Total genes (n_var): {combined_adata.n_vars}")
        print(f"      - Example obs_name: {combined_adata.obs_names[0]}")

    except Exception as e:
        print(f"    [!! ERROR !!] Merge failed for region '{region_name}': {e}")
        print("    Please check if files in this region are compatible (e.g., .var gene lists).")
        print("    Skipping this region.")
        continue # Move to the next region folder

    # --- 3.5 Save the combined file ---
    print(f"    Saving merged file to: {output_path}")
    try:
        # Use gzip compression to save space
        combined_adata.write_h5ad(output_path, compression='gzip')
        print(f"    Region '{region_name}' saved successfully!")
    except Exception as e:
        print(f"    [!! ERROR !!] Failed to save file: {e}")

    print(f"<<< Finished region: {region_name}")
    print("-" * 40)

print("\n===============================")
print("All region processing complete.")

In [ ]:
cb_chp = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_cb_chp.h5ad")
cb_chp